In [ ]:
import os
import requests
import zipfile
import glob
import random
import numpy as np
import pandas as pd
import rasterio
import torch
import cv2
import albumentations as A
from torch import nn
from torch.utils.data import Dataset
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    TrainingArguments,
    Trainer
)

# --- Experiment Configuration ---
class Config:
    # Experiment Identifiers
    EXPERIMENT_NAME = "segformer_b0_zenodo_burned_area"
    DATA_ROOT = "dataset"

    # Zenodo Data Sources
    DATA_URLS = [
        ("https://zenodo.org/record/6597139/files/satellite_data.csv?download=1", "satellite_data.csv"),
        ("https://zenodo.org/record/6597139/files/Satellite_burned_area_dataset_part1.zip?download=1", "Satellite_burned_area_dataset_part1.zip")
    ]

    # Data Processing
    CROP_SIZE = 512
    VALIDATION_FOLD = "purple" # Specific fold from the CSV to use for validation

    # Model Architecture
    MODEL_CHECKPOINT = "nvidia/segformer-b0-finetuned-ade-512-512"
    NUM_LABELS = 2
    ID2LABEL = {0: "Safe", 1: "Fire"}
    LABEL2ID = {"Safe": 0, "Fire": 1}

    # Training Hyperparameters
    BATCH_SIZE = 4
    LEARNING_RATE = 6e-5
    EPOCHS = 10 # Adjusted for standard run
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Reproducibility
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

print(f"⚙️ Configuration loaded for: {Config.EXPERIMENT_NAME}")
print(f"🔧 Device: {Config.DEVICE}")

In [ ]:
class DataManager:
    """
    Automates downloading and extracting the Zenodo Burned Area dataset.
    """
    @staticmethod
    def _download_file(url, save_path):
        print(f"⬇️ Downloading {save_path}...")
        response = requests.get(url, stream=True)
        with open(save_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024):
                if chunk: f.write(chunk)
        print("✅ Download complete.")

    @classmethod
    def setup_dataset(cls):
        os.makedirs(Config.DATA_ROOT, exist_ok=True)

        # 1. Download Files
        for url, filename in Config.DATA_URLS:
            file_path = os.path.join(Config.DATA_ROOT, filename)
            if not os.path.exists(file_path):
                cls._download_file(url, file_path)
            else:
                print(f"✅ Found {filename}, skipping download.")

        # 2. Extract Zip
        zip_path = os.path.join(Config.DATA_ROOT, "Satellite_burned_area_dataset_part1.zip")
        extract_path = os.path.join(Config.DATA_ROOT, "Satellite_burned_area_dataset_part1")

        # Check if already extracted (simple check for directory existence)
        if not os.path.exists(extract_path):
            print("📦 Extracting dataset (this may take a moment)...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(Config.DATA_ROOT)
            print("✅ Extraction complete.")
        else:
            print("✅ Dataset already extracted.")

# Trigger Setup
DataManager.setup_dataset()

Checking 65 candidates for train...
[TRAIN] Final Dataset: 15 valid samples ready.
Checking 8 candidates for val...
[VAL] Final Dataset: 2 valid samples ready.


/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:417: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


In [ ]:
class AdvancedBurnedAreaDataset(Dataset):
    """
    Robust Dataset for Zenodo Burned Area.
    Features:
    - Dynamic Band Selection (1-band duplication or RGB extraction)
    - Sniper Cropping (centers on fire pixels during training)
    - Robust Error Handling (skips corrupt files)
    """
    def __init__(self, csv_file, root_dir, feature_extractor, fold, split="train"):
        self.df = pd.read_csv(csv_file)
        self.root_dir = os.path.join(root_dir, "Satellite_burned_area_dataset_part1") # Adjust path to extracted folder
        self.feature_extractor = feature_extractor
        self.split = split
        self.crop_size = Config.CROP_SIZE

        # Filter by Fold
        if 'fold' in self.df.columns:
            if split == "train":
                self.df = self.df[self.df['fold'] != fold]
            else:
                self.df = self.df[self.df['fold'] == fold]

        # Validate file existence
        self._validate_files()

        # Transforms
        if split == "train":
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.RandomBrightnessContrast(p=0.2),
                # Ensure image is at least crop_size large
                A.PadIfNeeded(min_height=self.crop_size, min_width=self.crop_size, border_mode=0, value=0)
            ])
        else:
            # Validation: Strict Center Crop
            self.transform = A.Compose([
                A.PadIfNeeded(min_height=self.crop_size, min_width=self.crop_size, border_mode=0, value=0),
                A.CenterCrop(height=self.crop_size, width=self.crop_size)
            ])

    def _validate_files(self):
        """Pre-filter dataset to remove missing folders."""
        valid_indices = []
        for idx, row in self.df.iterrows():
            if os.path.exists(os.path.join(self.root_dir, row['folder'])):
                valid_indices.append(idx)
        self.df = self.df.loc[valid_indices].reset_index(drop=True)
        print(f"[{self.split.upper()}] Verified {len(self.df)} samples.")

    def __len__(self):
        return len(self.df)

    def _load_image_bands(self, img_path):
        """Handle 1-band vs Multi-band logic strictly."""
        with rasterio.open(img_path) as src:
            if src.count == 1:
                # Case 1: Single band -> Duplicate to 3 channels
                band = src.read(1)
                image = np.stack([band, band, band], axis=0)
            else:
                # Case 2: Multi-band -> Try RGB (4,3,2) or fallback (1,2,3)
                bands = [4, 3, 2] if src.count >= 4 else [1, 2, 3]
                # Safety check if even fallback bands exceed count
                if max(bands) > src.count:
                    bands = [1, 2, 3][:src.count]

                image = src.read(bands)

                # Pad if we still don't have 3 channels
                if image.shape[0] < 3:
                    pad = np.zeros((3 - image.shape[0], image.shape[1], image.shape[2]), dtype=image.dtype)
                    image = np.concatenate([image, pad], axis=0)

            # Normalize to 8-bit [0, 255]
            image = np.moveaxis(image, 0, -1) # (C, H, W) -> (H, W, C)
            if image.max() > 0:
                image = (image / image.max() * 255).astype(np.uint8)
            else:
                image = image.astype(np.uint8)
            return image

    def _sniper_crop(self, image, mask):
        """Training Logic: Attempt to center crop on fire pixels."""
        h, w = mask.shape
        # PadIfNeeded in transform ensures h, w >= crop_size

        fire_indices = np.argwhere(mask == 1)

        if len(fire_indices) > 0 and random.random() < 0.8:
            # Center on random fire pixel
            center_y, center_x = fire_indices[random.randint(0, len(fire_indices)-1)]
            top = max(0, min(center_y - self.crop_size//2, h - self.crop_size))
            left = max(0, min(center_x - self.crop_size//2, w - self.crop_size))
        else:
            # Random crop
            top = random.randint(0, h - self.crop_size)
            left = random.randint(0, w - self.crop_size)

        return image[top:top+self.crop_size, left:left+self.crop_size, :], \
               mask[top:top+self.crop_size, left:left+self.crop_size]

    def __getitem__(self, idx):
        # Retry loop to handle corrupt files gracefully
        attempts = 0
        while attempts < len(self):
            try:
                row = self.df.iloc[idx]
                folder_path = os.path.join(self.root_dir, row['folder'])

                # File Discovery
                all_files = os.listdir(folder_path)
                mask_files = [f for f in all_files if 'mask' in f.lower() and f.endswith(('.tif', '.tiff'))]
                # Prioritize Sentinel-2 images
                img_files = [f for f in all_files if f not in mask_files and f.endswith(('.tif', '.tiff')) and 'sentinel2' in f.lower()]

                if not img_files: # Fallback to any TIFF
                    img_files = [f for f in all_files if f not in mask_files and f.endswith(('.tif', '.tiff'))]

                if not img_files or not mask_files:
                    raise ValueError("Missing image or mask files.")

                img_files.sort()

                # Load Data
                image = self._load_image_bands(os.path.join(folder_path, img_files[-1]))

                with rasterio.open(os.path.join(folder_path, mask_files[0])) as src:
                    mask = src.read(1)
                    mask[mask > 0] = 1 # Binarize
                    mask = mask.astype(np.uint8)

                # Apply Albumentations
                augmented = self.transform(image=image, mask=mask)
                image = augmented['image']
                mask = augmented['mask']

                # Apply Sniper Crop (Train Only)
                if self.split == "train":
                    image, mask = self._sniper_crop(image, mask)

                # Final Safety Resize (rare fallback)
                if image.shape[0] != self.crop_size or image.shape[1] != self.crop_size:
                     image = cv2.resize(image, (self.crop_size, self.crop_size))
                     mask = cv2.resize(mask, (self.crop_size, self.crop_size), interpolation=cv2.INTER_NEAREST)

                # Encode for Model
                encoded = self.feature_extractor(
                    images=image,
                    segmentation_maps=mask,
                    return_tensors="pt",
                    do_resize=False,
                    do_rescale=False
                )

                return {k: v.squeeze() for k, v in encoded.items()}

            except Exception:
                # Move to next index if current fails
                idx = (idx + 1) % len(self.df)
                attempts += 1

        raise RuntimeError("Failed to load any valid samples after multiple attempts.")

In [ ]:
class ComboLoss(nn.Module):
    def __init__(self, smooth=1):
        super(ComboLoss, self).__init__()
        self.smooth = smooth
        self.ce = nn.CrossEntropyLoss(ignore_index=255)

    def forward(self, inputs, targets):
        # 1. Cross Entropy
        ce_loss = self.ce(inputs, targets)

        # 2. Dice Loss
        probs = torch.softmax(inputs, dim=1)
        fire_probs = probs[:, 1].contiguous().view(-1) # Class 1 = Fire
        targets_flat = targets.contiguous().view(-1)

        # Filter ignore_index (255) for Dice calc
        valid_mask = targets_flat != 255
        fire_probs = fire_probs[valid_mask]
        targets_flat = targets_flat[valid_mask]

        if len(targets_flat) == 0:
             return ce_loss

        intersection = (fire_probs * targets_flat).sum()
        dice = (2. * intersection + self.smooth) / (fire_probs.sum() + targets_flat.sum() + self.smooth)

        return 0.5 * ce_loss + 0.5 * (1 - dice)

class ComboTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.criterion = ComboLoss().to(Config.DEVICE)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Upsample logits to match labels (SegFormer outputs 1/4 res)
        logits_upsampled = nn.functional.interpolate(
            logits,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        loss = self.criterion(logits_upsampled, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# 1. Initialize Processor & Model
print(f"🚀 Loading Model: {Config.MODEL_CHECKPOINT}")
processor = SegformerImageProcessor.from_pretrained(
    Config.MODEL_CHECKPOINT,
    do_reduce_labels=False,
    do_resize=False,
    do_rescale=False
)

model = SegformerForSemanticSegmentation.from_pretrained(
    Config.MODEL_CHECKPOINT,
    num_labels=Config.NUM_LABELS,
    id2label=Config.ID2LABEL,
    label2id=Config.LABEL2ID,
    ignore_mismatched_sizes=True
)

# 2. Prepare Datasets
train_ds = AdvancedBurnedAreaDataset(
    os.path.join(Config.DATA_ROOT, "satellite_data.csv"),
    Config.DATA_ROOT,
    processor,
    fold=Config.VALIDATION_FOLD,
    split="train"
)
val_ds = AdvancedBurnedAreaDataset(
    os.path.join(Config.DATA_ROOT, "satellite_data.csv"),
    Config.DATA_ROOT,
    processor,
    fold=Config.VALIDATION_FOLD,
    split="val"
)

# 3. Define Arguments
args = TrainingArguments(
    output_dir=f"./results/{Config.EXPERIMENT_NAME}",
    learning_rate=Config.LEARNING_RATE,
    num_train_epochs=Config.EPOCHS,
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    fp16=(Config.DEVICE == "cuda"),
    remove_unused_columns=False,
    report_to="none"
)

# 4. Train
trainer = ComboTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print(f"🔥 Starting training on {Config.DEVICE}...")
trainer.train()

# 5. Save
save_path = f"./models/{Config.EXPERIMENT_NAME}"
trainer.save_model(save_path)
processor.save_pretrained(save_path)
print(f"🏆 Training finished. Model saved to {save_path}")